# Solution 04 — Formulate, diagnose, and repair an NLP

This worked solution implements every requested stage: a feasible bounded NLP,
all first-order KKT diagnostics, an intentionally infeasible equality, restoration
of the request, and an interactive parameter experiment. First try the exercise
predictions before using these cells as a reference.


## Setup

The checkout supplies generic diagnostics and plotting only. The objective,
constraint, derivatives, bounds, and solve calls are defined explicitly below.
The timer reports whole-solve elapsed time even when plugin timing fields are absent.


In [ ]:
# Locate the checkout without installing a model package.
import sys
from pathlib import Path

repository_root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'casadi_tutorial').is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import casadi as ca
import numpy as np
from time import perf_counter

from casadi_tutorial.diagnostics import bound_report, kkt_residuals, solver_summary
from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_series


## A. Write the mathematical expressions and solve-time interfaces

The objective is $f=(1-x_0)^2+100(x_1-x_0^2)^2$. The one-row constraint expression
is $g=x_0+x_1$; equality to $b$ is encoded numerically through identical `lbg/ubg`.
A symbolic `b` parameter is unnecessary for this expression because it appears
only in the numeric constraint bounds. This is an opportunity to distinguish
parameters inside a graph from data supplied as bounds.

The derivative function takes `x` and returns $\nabla f$, $J_g$, and $g$ in a fixed
order. The variable-bound multipliers are needed even when their small values
suggest those bounds are inactive.


In [ ]:
x = ca.MX.sym('exercise_x', 2)
f = (1-x[0])**2 + 100*(x[1]-x[0]**2)**2
g = x[0] + x[1]
derivatives = ca.Function('exercise_derivatives', [x],
                          [ca.gradient(f, x), ca.jacobian(g, x), g])
solver = ca.nlpsol('exercise_solver', 'ipopt', {'x': x, 'f': f, 'g': g}, {
    'print_time': False, 'error_on_fail': False,
    'ipopt.print_level': 0, 'ipopt.max_iter': 100, 'ipopt.tol': 1e-9,
})


### Solve and evaluate the candidate in the original formulation

`solve_target` retains status, solution, physical residuals, and labelled violations
for both success and failure. The function does not accept a failed candidate for
control or estimation. Bounds remain exactly those in the exercise. `lam_x` enters
stationarity and variable complementarity; omitting it would be incorrect if a
swept target activates a box bound.


In [ ]:
def solve_target(target=1.0):
    before = perf_counter()
    result = solver(x0=[0, 1], lbx=-2, ubx=2, lbg=target, ubg=target)
    elapsed = perf_counter() - before
    stats = solver.stats()
    gradient, jacobian, constraint = derivatives(result['x'])
    metrics = kkt_residuals(
        gradient, jacobian, result['lam_g'], constraint, target, target,
        variable=result['x'], lower_x=-2, upper_x=2, lambda_x=result['lam_x'],
    )
    rows = bound_report(constraint, target, target, labels=['sum equality'])
    rows += bound_report(result['x'], -2, 2, labels=['x[0]', 'x[1]'])
    return result, stats, solver_summary(stats, elapsed_s=elapsed), metrics, rows

solution, stats, summary, metrics, rows = solve_target()
print('candidate:', solution['x'], 'objective:', float(solution['f']))
print('equality multiplier:', solution['lam_g'], 'variable multipliers:', solution['lam_x'])
print(summary)
print('KKT:', metrics)
assert summary['success'] and np.isfinite(solution['x'].full()).all()
assert max(metrics.values()) < 1e-6


## B. Interpret the four diagnostics

`primal_inf` checks the equality and both variable bounds. `stationarity_inf`
checks $\nabla f+J_g^T\lambda_g+\lambda_x$. `complementarity_inf` checks multiplier
products with finite bound slacks; equality multipliers are unrestricted.
`dual_feasibility_inf` checks signs on one-sided bounds and zero multipliers on
fully unbounded rows. A proper two-sided interval permits either multiplier sign.

The box bounds are inactive for this target, so `lam_x` is small but generally
not bitwise zero. Tolerances account for numerical error. These local first-order
conditions do not constitute a global-minimum certificate for Rosenbrock's function.


## C. Prove, observe, and repair the impossible equality

For $b=5$, the box implies $x_0+x_1\le4$, contradicting the requested sum. The
solver should fail and leave significant primal infeasibility. We retain the
candidate only to diagnose it. Restoring $b=1$ repairs the operating request;
no solver-option change is needed.


In [ ]:
failed, failed_stats, failed_summary, failed_metrics, failed_rows = solve_target(5.0)
print('expected failure:', failed_summary)
print('failed KKT:', failed_metrics)
for row in sorted(failed_rows, key=lambda item: item['violation'], reverse=True):
    print(row)
assert not failed_summary['success']
assert failed_metrics['primal_inf'] > 0.1
repaired, _, repaired_summary, repaired_metrics, _ = solve_target(1.0)
assert repaired_summary['success'] and max(repaired_metrics.values()) < 1e-6
print('restored request:', repaired_summary, repaired_metrics)


## D. Plot convergence and sweep the requested sum

The residual plot uses solver iteration data where available; the default solution's
independent KKT report remains the acceptance test. A `1e-16` plotting floor makes
zero visible on log axes only. Each sweep target is feasible, and every result is
checked before recording it. Parameter sweeps can reveal active-set transitions;
they should not silently connect a failed result as though it were accepted.


In [ ]:
history = stats.get('iterations', {})
curves = {key: np.maximum(history[key], 1e-16) for key in ('inf_pr', 'inf_du') if key in history}
if curves:
    display_plot(plot_series(np.arange(len(next(iter(curves.values())))), curves,
                             xlabel='iteration', ylabel='reported infeasibility',
                             title='Feasible solve convergence', log_y=True))
targets = np.linspace(0, 2, 9)
objectives = []
for target in targets:
    result, _, info, residual, _ = solve_target(float(target))
    assert info['success'] and max(residual.values()) < 1e-5
    objectives.append(float(result['f']))
display_plot(plot_series(targets, {'optimal objective': objectives},
                         xlabel='requested sum b', ylabel='objective',
                         title='Feasible target sweep'))


### Optional live experiment

The normal solution and sweep already ran. This panel waits for an explicit click;
try a target outside `[-4,4]`, inspect the failure, then reset and rerun. A static
HTML page retains the default plots and needs no Python for hover or zoom, but
changing the optimization request requires a live JupyterLab kernel.


In [ ]:
def inspect_target(target):
    result, _, info, residual, report = solve_target(target)
    print(info, 'candidate:', result['x'], 'objective:', float(result['f']))
    print('KKT:', residual)
    for row in report:
        print(row)

experiment(inspect_target, {'target': 1.0}, title='Feasible versus infeasible requested sum')


## What the solution demonstrates

The constraint row ordering, bound multipliers, status, and physical residuals are
all visible. Contradictory requirements were established algebraically, reproduced
numerically, and repaired by restoring the request. Missing timing/history fields
are handled without inventing values. A useful next extension is to compare several
initial guesses at the same target and retain the best accepted local candidate.
